### 01 — Dataset Exploration and Preparation

What this notebook does:

- define the project data paths
- load and inspect the dataset annotations
- review the available clinical, image, and diagnostic fields
- identify missing or unavailable values
- verify that ultrasound images and tumour masks are present
- exclude normal cases so that the classification task is benign vs malignant
- separate the available variables according to their intended role
- encode the classification target
- create reproducible training, validation, and test splits
- save the processed splits for reuse by the later notebooks.

The saved splits are reused throughout the project so that segmentation, classification, multimodal fusion, and report generation are trained and evaluated on the same cases.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1. Define Project Paths

- `data/raw/labels.xlsx` contains the dataset annotations.
- `data/raw/images/` contains the ultrasound images and segmentation masks.
- `data/processed/` is used to store the train, validation, and test CSV files generated later in this notebook.


In [ ]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

LABEL_FILE = RAW_DIR / "labels.xlsx"
IMAGE_DIR = RAW_DIR / "images"

### 2. Load and Inspect the Clinical Dataset

The annotation spreadsheet is loaded into a Pandas DataFrame.


In [ ]:
df = pd.read_excel(LABEL_FILE)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print(df.columns.tolist())

In [ ]:
important_columns = [
    "CaseID",
    "Image_filename",
    "Mask_tumor_filename",
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms",
    "Shape",
    "Margin",
    "Echogenicity",
    "Posterior_features",
    "Halo",
    "Calcifications",
    "Skin_thickening",
    "Interpretation",
    "BIRADS",
    "Verification",
    "Diagnosis",
    "Classification"
]

display(df[important_columns].head())

In [ ]:
print(df["Classification"].value_counts(dropna=False))

### 3. Check Missing or Unavailable Data

The BrEaST spreadsheet does not store every missing clinical value as a standard value. Some entries are stored as text such as 'not available' or 'not applicable'

The following check searches explicitly for these values in addition to blank string. to show a overview of incomplete clinical information


In [ ]:
missing_strings = ["not available", "not applicable", ""]

for col in df.columns:
    count = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(missing_strings)
        .sum()
    )

    if count > 0:
        print(f"{col}: {count}")

### 4. Verify Image and Tumour-Mask Files

The notebook checks whether the filenames listed in the spreadsheet correspond to files that are actually present in the image directory.

This helps identify broken paths or missing files


In [ ]:
def file_exists(filename):
    if pd.isna(filename):
        return False

    return (IMAGE_DIR / str(filename)).exists()


df["image_exists"] = df["Image_filename"].apply(file_exists)

print(df["image_exists"].value_counts())

In [ ]:
def all_masks_exist(mask_value):
    if pd.isna(mask_value):
        return False

    filenames = str(mask_value).split("&")

    return all(
        (IMAGE_DIR / filename.strip()).exists()
        for filename in filenames
    )


df["tumor_mask_exists"] = df["Mask_tumor_filename"].apply(all_masks_exist)

print(df["tumor_mask_exists"].value_counts())

### 5. Remove Normal Cases

The original dataset has a small number of normal examinations. These cases are removed from the modelling subset because the project's main classification task is benign and malignant

Only records whose `Classification` value is either `benign` or `malignant` are retained in `df_model`.


In [ ]:
df_model = df[
    df["Classification"]
    .astype(str)
    .str.lower()
    .isin(["benign", "malignant"])
].copy()

print(df_model.shape)
print(df_model["Classification"].value_counts())

### 6. Group Variables by Their Intended Role

#### Patient clinical variables
These are the structured clinical inputs used by the ClinicalBERT branch:
- Age
- Tissue composition
- Signs
- Symptoms

#### Image descriptor variables
These describe lesion appearance in the ultrasound image, such as shape, margin, and echogenicity. They are used for analysis and report construction, but they are not used as direct classification inputs in the main multimodal model.

#### Diagnostic variables
These contain assessments or outcomes such as BI-RADS, diagnosis, verification, and final classification. They are kept separate because directly using them as classifier inputs could introduce target leakage.


In [ ]:
patient_clinical_columns = [
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms"
]

image_descriptor_columns = [
    "Shape",
    "Margin",
    "Echogenicity",
    "Posterior_features",
    "Halo",
    "Calcifications",
    "Skin_thickening"
]

diagnostic_columns = [
    "Interpretation",
    "BIRADS",
    "Verification",
    "Diagnosis",
    "Classification"
]

### 7. Encode the Binary Classification Target

The text labels are converted into numerical values for machine-learning use:
- benign → 0
- malignant → 1

The encoded value is stored in a new target column.


In [ ]:
df_model["target"] = (
    df_model["Classification"]
    .str.lower()
    .map({
        "benign": 0,
        "malignant": 1
    })
)

print(df_model[["Classification", "target"]].tail())

### 8. Create Training, Validation, and Test Splits

The dataset is divided into approximately:
- 70% training
- 15% validation
- 15% testing

The split is performed in two stages using train_test_split.
stratify=df_model["target"] preserves the benign/malignant class ratio across the subsets.
random_state=42 makes the split reproducible.

These splits are saved and reused in later notebooks so that all model components are trained and evaluated on a consistent partition of the data.


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.30,
    stratify=df_model["target"],
    random_state=42
)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

In [ ]:
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

In [ ]:
for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(split["Classification"].value_counts())

## 9. Save the Processed Dataset Splits

The three subsets are written to data/processed/ as:
- train.csv
- val.csv
- test.csv

These are the dataset partitions used by the segmentation, image-encoding, multimodal classification, report-generation, and final-evaluation notebooks.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)